In [3]:
import os
import random
trainval_percent = 0.1
train_percent = 0.9
xmlfilepath = 'train_data/labels'
txtsavepath = 'train_data/images'
total_xml = os.listdir(xmlfilepath)
num = len(total_xml)
list = range(num)
tv = int(num * trainval_percent)
tr = int(tv * train_percent)
trainval = random.sample(list, tv)
train = random.sample(trainval, tr)
ftrainval = open('train_data/ImageSets/trainval.txt', 'w')
ftest = open('train_data/ImageSets/test.txt', 'w')
ftrain = open('train_data/ImageSets/train.txt', 'w')
fval = open('train_data/ImageSets/val.txt', 'w')
for i in list:
    name = total_xml[i][:-4] + '\n'
    if i in trainval:
        ftrainval.write(name)
        if i in train:
            ftest.write(name)
        else:
            fval.write(name)
    else:
        ftrain.write(name)
ftrainval.close()
ftrain.close()
fval.close()
ftest.close()

In [6]:
import xml.etree.ElementTree as ET
import pickle
import os
from os import listdir, getcwd
from os.path import join
sets = ['train', 'test','val']
classes = ['1']
def convert(size, box):
    dw = 1. / size[0]
    dh = 1. / size[1]
    x = (box[0] + box[1]) / 2.0
    y = (box[2] + box[3]) / 2.0
    w = box[1] - box[0]
    h = box[3] - box[2]
    x = x * dw
    w = w * dw
    y = y * dh
    h = h * dh
    return (x, y, w, h)
def convert_annotation(image_id):
    in_file = open('train_data/labels/%s.xml' % (image_id))
    out_file = open('train_data/txt_labels/%s.txt' % (image_id), 'w')
    tree = ET.parse(in_file)
    root = tree.getroot()
    size = root.find('size')
    w = int(size.find('width').text)
    h = int(size.find('height').text)
    for obj in root.iter('object'):
        difficult = obj.find('difficult').text
        cls = obj.find('name').text
        if cls not in classes or int(difficult) == 1:
            continue
        cls_id = classes.index(cls)
        xmlbox = obj.find('bndbox')
        b = (float(xmlbox.find('xmin').text), float(xmlbox.find('xmax').text), float(xmlbox.find('ymin').text),
             float(xmlbox.find('ymax').text))
        bb = convert((w, h), b)
        out_file.write(str(cls_id) + " " + " ".join([str(a) for a in bb]) + '\n')
wd = getcwd()
print(wd)
for image_set in sets:
    if not os.path.exists('train_data/txt_labels/'):
        os.makedirs('train_data/txt_labels/')
    image_ids = open('train_data/ImageSets/%s.txt' % (image_set)).read().strip().split()
    list_file = open('train_data/%s.txt' % (image_set), 'w')
    for image_id in image_ids:
        list_file.write('train_data/images/%s.jpg\n' % (image_id))
        convert_annotation(image_id)
    list_file.close()

E:\Jupyter\OCR\train_code\train_ctpn


In [8]:
import cv2
import os

def change_labelimage_to_cptn_data(pictures_file_path, txt_file_path, cptn_data_labels_path):
    list = os.listdir(txt_file_path)
    # for txt_name in os.listdir(txt_file_path):
    #     txt_path = os.path.join(txt_file_path,txt_name).replace('\\','/')
    #     with open(txt_path,mode='rb') as f:
    #         data = f.readlines(txt_path)
    #         print(data)



    for i in range(1, len(list)+1):
        txt_path = txt_file_path + str(i).zfill(6) + '.txt'  #原数据集(label-image标注的)对应的txt标签存放路径
        dir = open(txt_path)
        lines = dir.readlines()
        lists = []  # 直接用一个数组存起来就好了
        for line in lines:
            lists.append(line.split())
        print(lists)
        im_path = pictures_file_path+str(i).zfill(6)+'.jpg'  #原数据集图片存放路径

        picture = cv2.imread(im_path)
         # print(len(lists))
        for j in range(0, len(lists)):
            a = lists[j][1]  #宽
            b = lists[j][2] #高
            c = lists[j][3] #宽度
            d = lists[j][4] #高度
            #根据txt标签中的归一化坐标还原为标注图像的真实坐标
            e = int((float(a) * picture.shape[1]) - (float(c) *int(picture.shape[1])/2))
            f = int((float(b) * picture.shape[0]) - (float(d) * picture.shape[0]/2))
            q = int((float(a) * picture.shape[1]) + (float(c) * picture.shape[1]/2))
            s = int((float(b) * picture.shape[0]) + (float(d) * picture.shape[0]/2))
            print(e,f,q,s)
            with open(cptn_data_labels_path + str(i).zfill(6)+'.txt', 'a+') as p:
                p.write(str(e) + ',' + str(f) + ',' + str(q) + ',' + str(s) + '\n')


if __name__ == '__main__':
    
    pictures_path = './train_data/images/' #图片路径
    txt_path = './train_data/txt_labels/' #label-image标注的标签路径
    ctpn_labels = './train_data/ctpn_labels/' #转换后，结果保存路径
    change_labelimage_to_cptn_data(pictures_path, txt_path, ctpn_labels)

[['0', '0.5058236272878536', '0.3207171314741036', '0.15973377703826955', '0.12350597609561753'], ['0', '0.5291181364392679', '0.598605577689243', '0.11314475873544093', '0.10557768924302789']]
256 130 352 192
284 274 352 327
[['0', '0.27848575712143925', '0.28455790784557905', '0.15067466266866567', '0.0958904109589041'], ['0', '0.7822338830584707', '0.4533001245330012', '0.15667166416791603', '0.09713574097135741']]
270 189 471 267
939 325 1148 403
[['0', '0.5538321167883211', '0.30363036303630364', '0.2062043795620438', '0.1254125412541254'], ['0', '0.583941605839416', '0.5338283828382838', '0.1678832116788321', '0.10396039603960396']]
247 146 360 222
274 292 366 355
[['0', '0.5593065693430657', '0.30363036303630364', '0.20255474452554745', '0.1254125412541254'], ['0', '0.583029197080292', '0.5387788778877888', '0.14781021897810218', '0.11056105610561057']]
251 146 362 222
279 293 360 360
[['0', '0.27199999999999996', '0.2659438775510204', '0.152', '0.08545918367346939'], ['0', '0.7

[['0', '0.5369127516778524', '0.2330508474576271', '0.23892617449664427', '0.10734463276836158'], ['0', '0.5355704697986577', '0.3418079096045198', '0.2442953020134228', '0.096045197740113'], ['0', '0.5375838926174497', '0.451271186440678', '0.25100671140939596', '0.10310734463276836']]
311 127 489 203
308 208 490 276
307 283 494 356
[['0', '0.5570175438596491', '0.22478991596638656', '0.23182957393483708', '0.10224089635854341'], ['0', '0.5607769423558897', '0.33753501400560226', '0.23433583959899748', '0.10364145658263306'], ['0', '0.5639097744360901', '0.44607843137254904', '0.23809523809523808', '0.09663865546218488']]
351 124 537 197
354 204 541 278
354 284 545 353
[['0', '0.5607769423558897', '0.22829131652661064', '0.23182957393483708', '0.10084033613445378'], ['0', '0.5620300751879699', '0.3403361344537815', '0.23182957393483708', '0.10644257703081232'], ['0', '0.56265664160401', '0.45028011204481794', '0.23558897243107768', '0.09943977591036415']]
355 127 540 199
355 205 541 2

271 189 467 267
950 326 1147 401
[['0', '0.5556569343065694', '0.30115511551155116', '0.19160583941605838', '0.11386138613861387'], ['0', '0.5894160583941606', '0.533003300330033', '0.14233576642335766', '0.09900990099009901']]
252 148 357 217
284 293 362 353
[['0', '0.2776666666666667', '0.26849489795918363', '0.142', '0.09056122448979591'], ['0', '0.7919999999999999', '0.40433673469387754', '0.14666666666666667', '0.08418367346938775']]
310 174 523 245
1078 284 1298 350
[['0', '0.27466666666666667', '0.26658163265306123', '0.14666666666666667', '0.09183673469387754'], ['0', '0.7923333333333333', '0.40561224489795916', '0.14866666666666667', '0.08673469387755102']]
302 173 522 245
1077 284 1300 352
[['0', '0.536241610738255', '0.23163841807909605', '0.2402684563758389', '0.1016949152542373'], ['0', '0.5355704697986577', '0.3425141242937853', '0.24966442953020132', '0.10028248587570622'], ['0', '0.5348993288590603', '0.4533898305084746', '0.2536912751677852', '0.1016949152542373']]
310

278 63 482 141
[['0', '0.5006596306068601', '0.22444444444444445', '0.2678100263852243', '0.17777777777777778']]
278 61 481 141
[['0', '0.5099833610648918', '0.3237051792828685', '0.15141430948419302', '0.11752988047808764'], ['0', '0.5282861896838602', '0.5956175298804781', '0.11480865224625625', '0.10756972111553785']]
261 133 352 192
283 272 352 326
[['0', '0.2788605697151424', '0.28580323785803236', '0.14842578710644677', '0.09339975093399751'], ['0', '0.7889805097451275', '0.45454545454545453', '0.14617691154422788', '0.0896637608966376']]
272 191 470 267
955 329 1150 401
[['0', '0.2788605697151424', '0.2876712328767123', '0.14392803598200898', '0.09464508094645081'], ['0', '0.7893553223388305', '0.45579078455790784', '0.14692653673163417', '0.08717310087173101']]
275 193 467 269
955 331 1151 401
[['0', '0.5602189781021898', '0.30198019801980197', '0.18248175182481752', '0.10561056105610561'], ['0', '0.5885036496350364', '0.5363036303630363', '0.14416058394160583', '0.102310231023

260 153 354 215
286 299 360 353
[['0', '0.5583941605839415', '0.30280528052805283', '0.18248175182481752', '0.10066006600660066'], ['0', '0.5866788321167883', '0.5371287128712872', '0.14051094890510948', '0.09075907590759076']]
255 153 355 214
283 298 360 353
[['0', '0.2786666666666667', '0.26785714285714285', '0.14533333333333334', '0.08928571428571427'], ['0', '0.7949999999999999', '0.4075255102040816', '0.146', '0.08035714285714285']]
309 175 527 245
1083 288 1302 351
[['0', '0.536241610738255', '0.2330508474576271', '0.2348993288590604', '0.10734463276836158'], ['0', '0.5369127516778524', '0.3446327683615819', '0.24966442953020132', '0.09887005649717515'], ['0', '0.5355704697986577', '0.4533898305084746', '0.24966442953020132', '0.09887005649717515']]
312 127 487 203
307 209 493 279
306 286 492 356
[['0', '0.5369127516778524', '0.22951977401129944', '0.2442953020134228', '0.1059322033898305'], ['0', '0.5335570469798657', '0.3418079096045198', '0.24563758389261744', '0.0932203389830

...

288 209 480 278
291 284 484 363
[['0', '0.5074875207986689', '0.32171314741035856', '0.15307820299500832', '0.12151394422310757'], ['0', '0.5232945091514143', '0.598605577689243', '0.11148086522462562', '0.10159362549800796']]
259 131 351 192
281 275 348 326
[['0', '0.56328320802005', '0.22899159663865545', '0.22932330827067668', '0.10224089635854341'], ['0', '0.5657894736842105', '0.3396358543417367', '0.24185463659147868', '0.09943977591036415'], ['0', '0.56328320802005', '0.45028011204481794', '0.24436090225563908', '0.09663865546218488']]
357 127 541 200
355 207 548 278
351 287 547 356
[['0', '0.27466666666666667', '0.27232142857142855', '0.144', '0.08545918367346939'], ['0', '0.7933333333333333', '0.4081632653061224', '0.156', '0.08418367346938775']]
304 179 520 246
1073 286 1307 352
[['0', '0.5520050125313283', '0.23319327731092437', '0.23684210526315788', '0.09663865546218488'], ['0', '0.5476190476190476', '0.33683473389355745', '0.23809523809523808', '0.10224089635854341'], ['0

279 62 487 144
[['0', '0.5456204379562044', '0.30445544554455445', '0.17518248175182483', '0.11056105610561057'], ['0', '0.5748175182481752', '0.5354785478547854', '0.14233576642335766', '0.08745874587458746']]
251 151 347 218
276 298 354 351
[['0', '0.26766666666666666', '0.26913265306122447', '0.146', '0.08418367346938775'], ['0', '0.7833333333333333', '0.4075255102040816', '0.14666666666666667', '0.08290816326530612']]
292 177 511 243
1065 287 1285 352
[['0', '0.5369127516778524', '0.2323446327683616', '0.24697986577181208', '0.1115819209039548'], ['0', '0.5322147651006711', '0.3453389830508475', '0.2402684563758389', '0.10875706214689265'], ['0', '0.5328859060402684', '0.4548022598870056', '0.24697986577181208', '0.096045197740113']]
308 125 492 204
307 206 486 283
305 288 489 356
[['0', '0.2754872563718141', '0.28829389788293897', '0.14767616191904048', '0.0958904109589041'], ['0', '0.7826086956521738', '0.4576587795765878', '0.14842578710644677', '0.0958904109589041']]
269 193 46

304 178 525 244
1074 289 1299 349
[['0', '0.49916805324459235', '0.32270916334661354', '0.13976705490848587', '0.10756972111553785'], ['0', '0.5174708818635607', '0.5926294820717132', '0.09983361064891848', '0.10557768924302789']]
258 135 342 189
281 271 341 324
[['0', '0.5620300751879699', '0.22408963585434175', '0.24686716791979949', '0.10644257703081232'], ['0', '0.56328320802005', '0.33683473389355745', '0.24686716791979949', '0.10224089635854341'], ['0', '0.5607769423558897', '0.4446778711484594', '0.24436090225563908', '0.10224089635854341']]
349 122 547 198
350 204 548 277
350 281 545 354
[['0', '0.5670426065162907', '0.22549019607843138', '0.23934837092731828', '0.11204481792717087'], ['0', '0.5651629072681704', '0.3396358543417367', '0.23809523809523808', '0.09943977591036415'], ['0', '0.5651629072681704', '0.44677871148459386', '0.24060150375939848', '0.10364145658263306']]
356 121 548 201
356 207 546 278
355 282 547 356
[['0', '0.5557644110275689', '0.22969187675070027', '0.

[['0', '0.5620437956204379', '0.3052805280528053', '0.1678832116788321', '0.09570957095709572'], ['0', '0.5848540145985401', '0.5346534653465347', '0.14051094890510948', '0.09240924092409242']]
262 156 354 214
282 296 359 352
[['0', '0.5529197080291971', '0.30363036303630364', '0.1897810218978102', '0.10231023102310231'], ['0', '0.5465328467153284', '0.5313531353135313', '0.19525547445255473', '0.10561056105610561']]
251 153 355 215
246 290 353 354
[['0', '0.27066666666666667', '0.2710459183673469', '0.14666666666666667', '0.08290816326530612'], ['0', '0.7863333333333333', '0.4068877551020408', '0.14333333333333334', '0.07653061224489795']]
296 180 516 245
1072 289 1287 349
[['0', '0.5105540897097625', '0.22444444444444445', '0.2717678100263852', '0.18666666666666668']]
283 59 489 143
[['0', '0.27599999999999997', '0.2704081632653061', '0.14266666666666666', '0.08418367346938775'], ['0', '0.7913333333333333', '0.4075255102040816', '0.14533333333333334', '0.08035714285714285']]
306 178 

[['0', '0.2773613193403298', '0.2876712328767123', '0.14392803598200898', '0.0921544209215442'], ['0', '0.7882308845577212', '0.45828144458281445', '0.14617691154422788', '0.0896637608966376']]
274 194 466 268
954 332 1149 404
[['0', '0.2786666666666667', '0.27232142857142855', '0.144', '0.09566326530612244'], ['0', '0.7969999999999999', '0.409438775510204', '0.14866666666666667', '0.07397959183673469']]
310 175 526 250
1084 291 1307 349
[['0', '0.56265664160401', '0.22478991596638656', '0.22807017543859648', '0.08823529411764706'], ['0', '0.5570175438596491', '0.3333333333333333', '0.23182957393483708', '0.10364145658263306'], ['0', '0.5551378446115288', '0.4432773109243697', '0.23558897243107768', '0.10504201680672269']]
357 129 540 192
351 201 537 275
349 279 537 354
[['0', '0.5116472545757071', '0.3247011952191235', '0.1314475873544093', '0.11553784860557768'], ['0', '0.5083194675540765', '0.5936254980079682', '0.05490848585690516', '0.09163346613545817']]
268 134 347 192
289 275 3

In [9]:
import os
import os.path
import shutil
#原始数据
fileDir_ann = r'E:/Jupyter/OCR/train_code/train_ctpn/train_data/labels/'
fileDir_img = r'E:/Jupyter/OCR/train_code/train_ctpn/train_data/images/'
 
#存放包含需要的类的图片
saveDir_img = r'E:/Jupyter/OCR/train_code/train_ctpn/train_data/img/'
        
if not os.path.exists(saveDir_img):
    os.mkdir(saveDir_img)
 
 
names = locals()
 
for files in os.walk(fileDir_ann):
    #遍历Annotations中的所有文件
    for file in files[2]:
        
 
        #存放包含需要的类的图片对应的xml文件
        saveDir_ann = r'E:/Jupyter/OCR/train_code/train_ctpn/train_data/xml/'
 
        if not os.path.exists(saveDir_ann):
            os.mkdir(saveDir_ann)
        fp = open(fileDir_ann + file)       
        saveDir_ann = saveDir_ann + file
        fp_w = open(saveDir_ann, 'w')
        classes = ['1']
 
        lines = fp.readlines()
 
        
        ind_start = []
 
        
        ind_end = []
 
        lines_id_start = lines[:]
        lines_id_end = lines[:]
 
        while "\t<object>\n" in lines_id_start:
            a = lines_id_start.index("\t<object>\n")
            ind_start.append(a)
            lines_id_start[a] = "delete"
 
        while "\t</object>\n" in lines_id_end:
            b = lines_id_end.index("\t</object>\n")
            ind_end.append(b)
            lines_id_end[b] = "delete"
 
        for k in range(0,len(ind_start)):
            for j in range(0,len(classes)):
                if classes[j] in lines[ind_start[k]+1]:
                    a = ind_start[k]
                    names['block%d'%k] = lines[a:ind_end[k]+1]
                    break
        #需要的类
        classes1 = '\t\t<name>1</name>\n'
 
        string_start = lines[0:ind_start[0]]
        string_end = lines[ind_end[-1] + 1:]
 
        a = 0
        for k in range(0,len(ind_start)):
            if classes1 in names['block%d'%k]:
                a += 1
                string_start += names['block%d'%k]
 
        string_start += string_end
        for c in range(0,len(string_start)):
            fp_w.write(string_start[c])
        fp_w.close()
 
        if a == 0:
            os.remove(saveDir_ann)
        else:
            name_img = fileDir_img + os.path.splitext(file)[0] + ".jpg"
            shutil.copy(name_img,saveDir_img)
        fp.close()

In [12]:
import xml.etree.ElementTree as ET
import os
import sys

if __name__ == "__main__":
    xmls_path = "E:/Jupyter/OCR/train_code/train_ctpn/train_data/xml"
    target_path = "E:/Jupyter/OCR/train_code/train_ctpn/train_data/txt"

    for xmlFilePath in os.listdir(xmls_path):
        print(os.path.join(xmls_path,xmlFilePath))
        try:
            tree = ET.parse(os.path.join(xmls_path,xmlFilePath))

            # 获得根节点
            root = tree.getroot()
        except Exception as e:  # 捕获除与程序退出sys.exit()相关之外的所有异常
            print("parse test.xml fail!")
            sys.exit()

        # objects = root.find("object")
        # print(len(objects))
        f = open(target_path +"/" + os.path.splitext(xmlFilePath)[0] + ".txt", 'w')
        # print(f)

 

        for bndbox in root.iter('bndbox'):
            node = []
            for child in bndbox:
                node.append(int(child.text))
            x1, y1 = node[0],node[1]
            x3, y3 = node[2],node[3]
    
            x2 ,y2 = x3 ,y1
            x4, y4 = x1, y3
        
            string = ''+str(x1)+','+str(y1)+','+str(x3)+','+str(y3)+','+str(x2)+','+str(y2)+','+str(x4)+','+str(y4);
            # print(string)
            f.write(string+'\n')

            
        f.close()

E:/Jupyter/OCR/train_code/train_ctpn/train_data/xml\000001.xml
E:/Jupyter/OCR/train_code/train_ctpn/train_data/xml\000002.xml
E:/Jupyter/OCR/train_code/train_ctpn/train_data/xml\000003.xml
E:/Jupyter/OCR/train_code/train_ctpn/train_data/xml\000004.xml
E:/Jupyter/OCR/train_code/train_ctpn/train_data/xml\000005.xml
E:/Jupyter/OCR/train_code/train_ctpn/train_data/xml\000006.xml
E:/Jupyter/OCR/train_code/train_ctpn/train_data/xml\000007.xml
E:/Jupyter/OCR/train_code/train_ctpn/train_data/xml\000008.xml
E:/Jupyter/OCR/train_code/train_ctpn/train_data/xml\000009.xml
E:/Jupyter/OCR/train_code/train_ctpn/train_data/xml\000010.xml
E:/Jupyter/OCR/train_code/train_ctpn/train_data/xml\000011.xml
E:/Jupyter/OCR/train_code/train_ctpn/train_data/xml\000012.xml
E:/Jupyter/OCR/train_code/train_ctpn/train_data/xml\000013.xml
E:/Jupyter/OCR/train_code/train_ctpn/train_data/xml\000014.xml
E:/Jupyter/OCR/train_code/train_ctpn/train_data/xml\000015.xml
E:/Jupyter/OCR/train_code/train_ctpn/train_data/xml\000

E:/Jupyter/OCR/train_code/train_ctpn/train_data/xml\000247.xml
E:/Jupyter/OCR/train_code/train_ctpn/train_data/xml\000248.xml
E:/Jupyter/OCR/train_code/train_ctpn/train_data/xml\000249.xml
E:/Jupyter/OCR/train_code/train_ctpn/train_data/xml\000250.xml
E:/Jupyter/OCR/train_code/train_ctpn/train_data/xml\000251.xml
E:/Jupyter/OCR/train_code/train_ctpn/train_data/xml\000252.xml
E:/Jupyter/OCR/train_code/train_ctpn/train_data/xml\000253.xml
E:/Jupyter/OCR/train_code/train_ctpn/train_data/xml\000254.xml
E:/Jupyter/OCR/train_code/train_ctpn/train_data/xml\000255.xml
E:/Jupyter/OCR/train_code/train_ctpn/train_data/xml\000256.xml
E:/Jupyter/OCR/train_code/train_ctpn/train_data/xml\000257.xml
E:/Jupyter/OCR/train_code/train_ctpn/train_data/xml\000258.xml
E:/Jupyter/OCR/train_code/train_ctpn/train_data/xml\000259.xml
E:/Jupyter/OCR/train_code/train_ctpn/train_data/xml\000260.xml
E:/Jupyter/OCR/train_code/train_ctpn/train_data/xml\000261.xml
E:/Jupyter/OCR/train_code/train_ctpn/train_data/xml\000

E:/Jupyter/OCR/train_code/train_ctpn/train_data/xml\000480.xml
E:/Jupyter/OCR/train_code/train_ctpn/train_data/xml\000481.xml
E:/Jupyter/OCR/train_code/train_ctpn/train_data/xml\000482.xml
E:/Jupyter/OCR/train_code/train_ctpn/train_data/xml\000483.xml
E:/Jupyter/OCR/train_code/train_ctpn/train_data/xml\000484.xml
E:/Jupyter/OCR/train_code/train_ctpn/train_data/xml\000485.xml
E:/Jupyter/OCR/train_code/train_ctpn/train_data/xml\000486.xml
E:/Jupyter/OCR/train_code/train_ctpn/train_data/xml\000487.xml
E:/Jupyter/OCR/train_code/train_ctpn/train_data/xml\000488.xml
E:/Jupyter/OCR/train_code/train_ctpn/train_data/xml\000489.xml
E:/Jupyter/OCR/train_code/train_ctpn/train_data/xml\000490.xml
E:/Jupyter/OCR/train_code/train_ctpn/train_data/xml\000491.xml
E:/Jupyter/OCR/train_code/train_ctpn/train_data/xml\000492.xml
E:/Jupyter/OCR/train_code/train_ctpn/train_data/xml\000493.xml
E:/Jupyter/OCR/train_code/train_ctpn/train_data/xml\000494.xml
E:/Jupyter/OCR/train_code/train_ctpn/train_data/xml\000

In [2]:
import os
import sys
import cv2

img_path = 'E:/Jupyter/OCR/train_code/train_ctpn/train_data/train_img/'
gt_path = 'E:/Jupyter/OCR/train_code/train_ctpn/train_data/gt/'


def read_path(file_pathname):
    #遍历该目录下的所有图片文件
    for filename in os.listdir(file_pathname):
        #print(filename)
        img = cv2.imread(file_pathname+'/'+filename)
        new_image = cv2.resize(img, (960,960), interpolation=cv2.INTER_AREA)
        #####保存图片的路径
        cv2.imwrite(file_pathname+"/"+filename, new_image)
        
read_path(img_path)